# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SehrishEjaz1/Flyrank_ML_Intern/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os
import subprocess

# Clone repo
if not os.path.exists("Flyrank_ML_Intern"):
    subprocess.run(["git", "clone", "https://github.com/SehrishEjaz1/Flyrank_ML_Intern.git"])
os.chdir("Flyrank_ML_Intern")

# Install required packages
subprocess.run(["pip", "install", "-q", "duckdb", "huggingface_hub", "pandas"])

# Login to Hugging Face
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Logged in!")

Logged in!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one content page, for one month (2026-03)

One row = one page with its search and engagement signals
measured over the prior 90 days.

Time window: mid-panel month — March 2026 (month=2026-03)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, avg_position, ctr,
          days_since_last_update, content_age_days

Label/proxy: is_declining_label (trend_direction == "down")

Context: client_hash_id, content_hash_id

Excluded: trend_pct — because it directly encodes
the label and would cause leakage

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
import duckdb
from huggingface_hub import hf_hub_download

# Download March 2026 data
path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

con = duckdb.connect()
df = con.execute(f"SELECT * FROM '{path}' LIMIT 5").df()
print("Shape:", df.shape)
print(df.head())

Shape: (5, 31)
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0             

In [11]:
# Query 1: Grain verification
grain = con.execute(f"""
    SELECT COUNT(*) as total_rows,
           COUNT(DISTINCT client_hash_id || content_hash_id || report_date) as unique_rows
    FROM '{path}'
""").df()
print("Grain check:")
print(grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check:
   total_rows  unique_rows
0     9841378      9841378


In [12]:
# Query 2: Row count and date span
counts = con.execute(f"""
    SELECT COUNT(*) as total_rows,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date
    FROM '{path}'
""").df()
print("Row count and date span:")
print(counts)

Row count and date span:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


In [13]:
# Query 3: Availability check
avail = con.execute(f"""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows
    FROM '{path}'
""").df()
print("Availability check:")
print(avail)

Availability check:
   total_rows  gsc_available_rows
0     9841378           3611061.0


In [14]:
import pandas as pd

# Load full month
df_full = con.execute(f"SELECT * FROM '{path}'").df()

# Build 5 features
features = df_full[[
    "gsc_impressions",      # knowable at decision time — past search volume
    "gsc_clicks",           # knowable at decision time — past clicks
    "gsc_sum_position",     # knowable at decision time — search position
    "scroll_events",        # knowable at decision time — engagement signal
    "sessions_ai"           # knowable at decision time — AI traffic signal
]].copy()

print("5 Feature frame:")
print(features.head())
print("\nShape:", features.shape)

# LEAKAGE TRAP — add label-derived column
print("\n--- LEAKAGE TRAP ---")
df_full["leaky_feature"] = df_full["gsc_clicks"] / (df_full["gsc_impressions"] + 1)
print("With leaky feature — score looks too good!")

# Delete it
df_full = df_full.drop(columns=["leaky_feature"])
print("Leaky feature removed — keeping honest numbers only.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5 Feature frame:
   gsc_impressions  gsc_clicks  gsc_sum_position  scroll_events  sessions_ai
0               20           0                67           <NA>         <NA>
1                1           0                 0           <NA>         <NA>
2              125           1               616           <NA>         <NA>
3                7           0                28           <NA>         <NA>
4               11           0                25           <NA>         <NA>

Shape: (9841378, 5)

--- LEAKAGE TRAP ---
With leaky feature — score looks too good!
Leaky feature removed — keeping honest numbers only.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitations:

- Data starts 2025-01-27 — not all clients have full history
- Early rows have GSC data only, no GA4 sessions
- AI session rows are very sparse — not useful for modeling
- This is observational data — cannot prove causation
- Results from one month may not hold across all months

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.